# Deteksi Postingan Palsu Reddit Multimodal
Notebook ini melakukan deteksi berita palsu menggunakan data multimodal (teks + gambar) dari Reddit (dari dataset [Fakeddit](https://https://fakeddit.netlify.app/)).

**File yang dibutuhkan:**
- train_multimodal.tsv
- test_multimodal.tsv
- images_data.zip

**Langkah-langkah:**

Membaca data teks dan gambar dari file TSV dan folder gambar


---


Metode 1:




- Ekstraksi fitur sederhana dari teks dan gambar
- Early fusion dengan menggabungkan fitur
- Klasifikasi dengan model machine learning


---


Metode 2:
- Ekstraksi fitur dengan BERT dan ResNet
- Klasifikasi dengan neural network



In [ ]:
import os

# Ekstraksi folder gambar dulu

!unzip -q images_data.zip

In [ ]:
import pandas as pd

# Membaca file TSV
train_df = pd.read_csv('train_multimodal.tsv', sep='\t')
test_df = pd.read_csv('test_multimodal.tsv', sep='\t')

# Menampilkan contoh data
train_df.head()

## EDA


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
import os

# Informasi umum
print(f"\nJumlah data latih: {len(train_df)}")
print(f"Jumlah data uji: {len(test_df)}")

# Visualisasi distribusi label
plt.figure(figsize=(5, 4))
sns.countplot(x='label', data=train_df)
plt.title('Distribusi Label pada Data Latih')
plt.xlabel('Label (0 = Fake, 1 = Asli)')
plt.ylabel('Jumlah')
plt.show()






### Analisis panjang teks

In [ ]:
# Analisis panjang teks


train_df['text_length'] = train_df['text'].apply(lambda x: len(x.split()))

plt.figure(figsize=(6, 4))
sns.histplot(data=train_df, x='text_length', hue='label', bins=20, kde=False)
plt.title('Distribusi Panjang Teks berdasarkan Label')
plt.xlabel('Jumlah Kata')
plt.ylabel('Frekuensi')
plt.show()




In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import textwrap

def tampilkan_contoh_grid(df, label, folder='images_data'):
    contoh = df[df['label'] == label].sample(6)

    fig, axes = plt.subplots(2, 3, figsize=(9, 9))
    fig.suptitle(f"Contoh Postingan Label {label} ({'Asli' if label == 1 else 'Palsu'})", fontsize=14)

    axes = axes.flatten()

    for ax, (_, row) in zip(axes, contoh.iterrows()):
        img_path = os.path.join(folder, f"{row['id']}.jpg")

        if os.path.exists(img_path):
            img = Image.open(img_path).resize((128, 128))
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "Gambar tidak ditemukan", ha='center', va='center')

        ax.axis('off')
        ax.set_title(f"ID {row['id']}", fontsize=10)

        # Wrap and show text below image
        wrapped_text = "\n".join(textwrap.wrap(row['text'], width=40))
        ax.text(0.5, -0.1, wrapped_text, ha='center', va='top', fontsize=8, transform=ax.transAxes)

    plt.tight_layout(rect=[0, 0, 1, 1.05])
    plt.show()

# Tampilkan contoh dari masing-masing label
tampilkan_contoh_grid(train_df, label=0)
print("\n-------------------------------------------------------------------------------------------\n")
tampilkan_contoh_grid(train_df, label=1)


### Praproses Gambar
Resize semua gambar ke 64x64 dan ekstrak histogram warna sebagai fitur sederhana.

In [ ]:
import numpy as np
from PIL import Image
import os
from tqdm import tqdm

def ekstrak_fitur_gambar(ids, folder='images_data'):
    fitur = []
    for id_ in tqdm(ids):
        path = os.path.join(folder, f"{id_}.jpg")
        try:
            img = Image.open(path).convert('RGB').resize((64, 64))  # pastikan RGB
            hist = np.array(img.histogram()) / 255.0
        except:
            hist = np.zeros(256 * 3)  # fallback jika gagal
        fitur.append(hist)
    return np.array(fitur)


# Ekstraksi fitur gambar
train_img_feat = ekstrak_fitur_gambar(train_df['id'])
test_img_feat = ekstrak_fitur_gambar(test_df['id'])



In [ ]:
train_img_feat.shape

### Praproses Teks
Gunakan TF-IDF untuk mengekstrak fitur dari teks.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=300)
train_txt_feat = tfidf.fit_transform(train_df['text']).toarray()
test_txt_feat = tfidf.transform(test_df['text']).toarray()

In [ ]:
train_txt_feat.shape

### Early Fusion: Model Logistic Regression
Gabungkan fitur teks dan gambar, lalu gunakan Logistic Regression.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train = np.hstack([train_txt_feat, train_img_feat])
X_test = np.hstack([test_txt_feat, test_img_feat])
y_train = train_df['label']
y_test = test_df['label']

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

### Early Fusion: Model Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Buat model Random Forest
rf = RandomForestClassifier(n_estimators=150, random_state=42)
rf.fit(X_train, y_train)

# Prediksi
y_pred2 = rf.predict(X_test)

# Laporan hasil klasifikasi
print(classification_report(y_test, y_pred2))


###  Pertanyaan Analisis
- Model mana yang memberikan hasil lebih baik? Mengapa?
- Apa kelebihan dan kelemahan fitur TF-IDF dan histogram gambar?


## Metode lebih canggih
Selanjutnya kita mencoba ekstrasi fitur teks dengan BERT serta ekstraksi fitur gambar dengan ResNet50 (pretrained).
Kedua sumber fitur digabungkan dan dilakukan klasifikasi dengan deep learning sederhana.

### Ekstraksi Fitur (BERT dan ResNet50)

In [ ]:
# Install dependensi
!pip install -q transformers torchvision

from transformers import BertTokenizer, BertModel
import torch
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load tokenizer dan model BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)
bert_model.eval()

# Fungsi ekstraksi fitur teks menggunakan BERT
def ekstrak_fitur_bert(teks_list):
    features = []
    with torch.no_grad():
        for teks in tqdm(teks_list):
            inputs = tokenizer(teks, return_tensors='pt', truncation=True, padding=True, max_length=64)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = bert_model(**inputs)
            cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()
            features.append(cls_embedding)
    return np.array(features)

# Ekstrak fitur teks
train_bert_feat = ekstrak_fitur_bert(train_df['text'])
test_bert_feat = ekstrak_fitur_bert(test_df['text'])

# Load pretrained ResNet50
resnet = models.resnet50(pretrained=True)
resnet.fc = torch.nn.Identity()  # Remove final classification layer
resnet = resnet.to(device)
resnet.eval()

# Transformasi gambar
img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Fungsi ekstraksi fitur gambar dengan ResNet50
def ekstrak_fitur_resnet(ids, folder='images_data'):
    fitur = []
    with torch.no_grad():
        for id_ in tqdm(ids):
            path = os.path.join(folder, f"{id_}.jpg")
            try:
                img = Image.open(path).convert('RGB')
                img_tensor = img_transform(img).unsqueeze(0).to(device)
                output = resnet(img_tensor).squeeze().cpu().numpy()
            except:
                output = np.zeros(2048)
            fitur.append(output)
    return np.array(fitur)

# Ekstrak fitur gambar
train_resnet_feat = ekstrak_fitur_resnet(train_df['id'])
test_resnet_feat = ekstrak_fitur_resnet(test_df['id'])

# Gabungkan fitur teks dan gambar
X_train_fusion = np.concatenate([train_bert_feat, train_resnet_feat], axis=1)
X_test_fusion = np.concatenate([test_bert_feat, test_resnet_feat], axis=1)
y_train = train_df['label'].values
y_test = test_df['label'].values


In [ ]:
X_train_fusion.shape

### Klasifikasi dengan Arsitektur Neural Network Sederhana

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Bangun model deep learning sederhana
model = models.Sequential([
    layers.Input(shape=(X_train_fusion.shape[1],)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Latih model
history = model.fit(X_train_fusion, y_train, validation_data=(X_test_fusion, y_test), epochs=10, batch_size=16)

# Evaluasi
loss, acc = model.evaluate(X_test_fusion, y_test)
print(f"Akurasi: {acc:.2f}")


### **Soal**:
Lakukan training dan uji sel di atas (neural network)sebanyak 10 kali kemudian rata-ratakan.

## Bonus
Pelajari terkait late fusion. Lakukan eksperimen klasifikasi multimodal dengan late fusion (metode ekstraksi fitur dan algoritma klasifikasi bebas).

In [ ]:
## Kerjakan di sini